# leenef Colab launcher

This notebook is intentionally thin. It clones the repository, installs dependencies, runs the checked-in smoke test script, and saves a JSON report.

Use this first to verify that the Colab VM, Python environment, and selected device all work before running larger suites.

In [ ]:
REPO_URL = "https://github.com/nikeke/leenef.git"
REPO_REF = "main"
WORKDIR = "/content/leenef"
USE_DRIVE = False
DRIVE_ROOT = "/content/drive/MyDrive/leenef-colab"

SMOKE_ARGS = {
    "device": "auto",
    "mnist_train": 512,
    "mnist_test": 256,
    "mnist_neurons": 256,
    "stream_train": 512,
    "stream_test": 256,
    "stream_neurons": 256,
    "stream_window": 5,
    "stream_batch": 64,
}

SUITE = "row_focus"  # or "sequential_hard"
SUITE_QUICK = False
LSTM_EPOCHS = 20


In [ ]:
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS_ROOT = Path(DRIVE_ROOT)
else:
    RESULTS_ROOT = Path(WORKDIR) / "results"

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_JSON = RESULTS_ROOT / "colab_smoke.json"
OUTPUT_JSON

In [ ]:
import shutil
from pathlib import Path

workdir = Path(WORKDIR)
if workdir.exists():
    shutil.rmtree(workdir)

!git clone {REPO_URL} {WORKDIR}
%cd {WORKDIR}
!git checkout {REPO_REF}
!python -m pip install -q --upgrade pip
!python -m pip install -q -e '.[dev,bench]'


In [ ]:
import platform
import torch

print("Python:", platform.python_version())
print("Platform:", platform.platform())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))


In [ ]:
%cd {WORKDIR}
!python benchmarks/colab_smoke.py \
    --device {SMOKE_ARGS['device']} \
    --mnist-train {SMOKE_ARGS['mnist_train']} \
    --mnist-test {SMOKE_ARGS['mnist_test']} \
    --mnist-neurons {SMOKE_ARGS['mnist_neurons']} \
    --stream-train {SMOKE_ARGS['stream_train']} \
    --stream-test {SMOKE_ARGS['stream_test']} \
    --stream-neurons {SMOKE_ARGS['stream_neurons']} \
    --stream-window {SMOKE_ARGS['stream_window']} \
    --stream-batch {SMOKE_ARGS['stream_batch']} \
    --output {OUTPUT_JSON}


In [ ]:
import json
from pathlib import Path

payload = json.loads(Path(OUTPUT_JSON).read_text())
payload


## Next step

Once the smoke test passes, run one of the checked-in suite scripts. `row_focus` is the first recommended Colab run; `sequential_hard` is the longer-sequence follow-up.

In [ ]:
quick_flag = "--quick" if SUITE_QUICK else ""
%cd {WORKDIR}
!python benchmarks/colab_suites.py --suite {SUITE} --device {SMOKE_ARGS['device']} --output-dir {RESULTS_ROOT} --lstm-epochs {LSTM_EPOCHS} {quick_flag}


The suite runner writes both JSON and CSV outputs under `RESULTS_ROOT`. If you use Drive, those files survive VM restarts and are easy to share back.